In [0]:
%pip install xgboost mlflow scikit-learn
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
import pandas as pd

df = spark.table('gold.ml_features').toPandas()

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nCancer diagnosis values: {df['lung_cancer_diagnosis'].unique()}")
print(f"\nMissing values:\n{df.isnull().sum()}")

In [0]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

df['target'] = (df['lung_cancer_diagnosis'] == 'Yes').astype(int)

le = LabelEncoder()
df['gender_enc']  = le.fit_transform(df['gender'])
df['country_enc'] = le.fit_transform(df['country'])
df['smoker_enc']  = le.fit_transform(df['smoker'])
df['passive_enc'] = le.fit_transform(df['passive_smoker'])
df['family_enc']  = le.fit_transform(df['family_history'])
df['airpol_enc']  = le.fit_transform(df['air_pollution_exposure'])
df['occup_enc']   = le.fit_transform(df['occupational_exposure'])
df['indoor_enc']  = le.fit_transform(df['indoor_pollution'])

FEATURES = [
    'age', 'gender_enc', 'country_enc', 'smoker_enc',
    'years_of_smoking', 'cigarettes_per_day',
    'passive_enc', 'family_enc',
    'airpol_enc', 'occup_enc', 'indoor_enc'
]

X = df[FEATURES]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"Cancer rate: {y.mean():.3f}")

In [0]:
import xgboost as xgb
import matplotlib.pyplot as plt
import pandas as pd

correlations = X.copy()
correlations['target'] = y

corr = correlations.corr()['target'].drop('target').sort_values(ascending=False)
print("Feature correlation with lung cancer diagnosis:")
print(corr)

In [0]:
import mlflow
import mlflow.xgboost
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, classification_report

FEATURES_FINAL = [
    'smoker_enc',
    'cigarettes_per_day',
    'years_of_smoking',
    'gender_enc'

]

X_train_f = X_train[FEATURES_FINAL]
X_test_f  = X_test[FEATURES_FINAL]

mlflow.set_experiment('/lung_cancer_risk_prediction')

with mlflow.start_run(run_name='xgboost_final'):

    params = {
        'n_estimators':     300,
        'max_depth':        5,
        'learning_rate':    0.05,
        'subsample':        0.8,
        'colsample_bytree': 0.8,
        'random_state':     42,
        'scale_pos_weight': len(y_train[y_train==0]) / len(y_train[y_train==1])
    }

    mlflow.log_params(params)
    mlflow.log_param('features', FEATURES_FINAL)

    model = xgb.XGBClassifier(**params)
    model.fit(X_train_f, y_train, eval_set=[(X_test_f, y_test)], verbose=False)

    y_pred = model.predict(X_test_f)
    y_prob = model.predict_proba(X_test_f)[:, 1]

    auc = roc_auc_score(y_test, y_prob)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred)

    mlflow.log_metric('auc',      auc)
    mlflow.log_metric('accuracy', acc)
    mlflow.log_metric('f1',       f1)

    mlflow.xgboost.log_model(model, 'xgboost_model')

    run_id = mlflow.active_run().info.run_id
    print(f'Run ID: {run_id}')
    print(f'AUC:      {auc:.4f}')
    print(f'Accuracy: {acc:.4f}')
    print(f'F1:       {f1:.4f}')
    print(classification_report(y_test, y_pred))

In [0]:
from mlflow.models.signature import infer_signature

signature = infer_signature(X_train_f, model.predict(X_train_f))

model_uri = f'runs:/{run_id}/xgboost_model'

with mlflow.start_run(run_name='xgboost_final_signed'):
    mlflow.xgboost.log_model(
        model,
        'xgboost_model',
        signature=signature,
        input_example=X_train_f.iloc[:5]
    )
    run_id_signed = mlflow.active_run().info.run_id

registered = mlflow.register_model(
    f'runs:/{run_id_signed}/xgboost_model',
    'LungCancerRiskModel'
)
print(f'Model registered: version {registered.version}')

In [0]:
predictions = model.predict_proba(X_test_f)[:, 1]

df_preds = X_test_f.copy()
df_preds['actual']          = y_test.values
df_preds['predicted_prob']  = predictions
df_preds['predicted_label'] = (predictions >= 0.3).astype(int)

spark.createDataFrame(df_preds) \
    .write.format('delta').mode('overwrite') \
    .saveAsTable('gold.risk_predictions')

print(f"Saved {len(df_preds):,} predictions to gold.risk_predictions")